In [ ]:
import cv2
import numpy as np
import os
import csv
import time
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO

# Settings
%matplotlib inline
plt.rcParams['figure.figsize'] = [16, 10]
plt.style.use('seaborn-v0_8-whitegrid')

# Paths
INPUT_DIR = "/Users/veeralpatel/vehicle-speed-estimation-dip/content"
OUTPUT_DIR = "/Users/veeralpatel/vehicle-speed-estimation-dip/content/processed/cascaded"
VIDEO_NAME = "cctv052x2004080516x01640.avi" 
INPUT_VIDEO_PATH = os.path.join(INPUT_DIR, VIDEO_NAME)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Constants
CONF_THRESHOLD = 0.50
FRAME_WIDTH_M = 30
FRAME_HEIGHT_M = 100
SOURCE_POLYGON = np.array([[20, 200], [300, 220], [280, 100], [40, 80]], dtype=np.float32)
BIRD_EYE_VIEW = np.array([[0, 0], [FRAME_WIDTH_M, 0], [FRAME_WIDTH_M, FRAME_HEIGHT_M], [0, FRAME_HEIGHT_M]], dtype=np.float32)
TRANSFORM_MATRIX = cv2.getPerspectiveTransform(SOURCE_POLYGON, BIRD_EYE_VIEW)

print(f"Pipeline Configured. Outputting to: {OUTPUT_DIR}")

In [ ]:
def get_dark_channel(image, size=15):
    min_channel = np.min(image, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (size, size))
    dark_channel = cv2.erode(min_channel, kernel)
    return dark_channel

def get_atmospheric_light(image, dark_channel):
    h, w = image.shape[:2]
    image_size = h * w
    num_pixels = int(max(math.floor(image_size / 1000), 1))
    dark_vec = dark_channel.reshape(image_size)
    image_vec = image.reshape(image_size, 3)
    indices = dark_vec.argsort()[-num_pixels:]
    atoms = np.mean(image_vec[indices], axis=0)
    return atoms

def get_transmission(image, atmosphere, omega=0.95, size=15):
    normalized = image / atmosphere
    transmission = 1 - omega * get_dark_channel(normalized, size)
    return transmission

def apply_dehaze(frame):
    img_f = frame.astype(np.float32) / 255.0
    dark = get_dark_channel(img_f, size=15)
    A = np.percentile(dark, 99)
    t = 1.0 - 0.95 * dark
    t = np.clip(t, 0.1, 1.0) 
    J = (img_f - A) / cv2.merge([t, t, t]) + A
    J = np.clip(J, 0, 1)
    return (J * 255).astype(np.uint8)

def apply_median_denoise(frame):
    return cv2.medianBlur(frame, 5)

def apply_adaptive_gamma(frame):
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y = img_yuv[:, :, 0]
    mean_bright = np.mean(y) + 1e-5
    gamma = np.log(128/255) / np.log(mean_bright/255)
    gamma = np.clip(gamma, 0.5, 2.5)
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    img_yuv[:, :, 0] = cv2.LUT(y, table)
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_clahe(frame):
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_contrast_stage(frame):
    stage1 = apply_adaptive_gamma(frame)
    return apply_clahe(stage1)

def apply_cascaded_pipeline(frame):
    dehazed = apply_dehaze(frame)
    denoised = apply_median_denoise(dehazed)
    final = apply_contrast_stage(denoised)
    return final

In [ ]:

cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
ret, frame = cap.read()
cap.release()

if ret:
    bad_frame = cv2.convertScaleAbs(frame, alpha=0.6, beta=-40) # Dark/Low Contrast
    noise = np.random.normal(0, 20, bad_frame.shape).astype(np.uint8) # Noise
    bad_frame = cv2.add(bad_frame, noise)
    step1_dehaze = apply_dehaze(bad_frame)
    step2_denoise = apply_median_denoise(step1_dehaze)
    step3_contrast = apply_contrast_stage(step2_denoise) # Final
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    
    images = [bad_frame, step1_dehaze, step2_denoise, step3_contrast]
    titles = ['1. Input (Haze+Noise+Dark)', '2. Dehazed (Artifacts appear)', '3. Denoised (Cleaned)', '4. Final (Contrast Enhanced)']
    
    for i, img in enumerate(images):
        axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(titles[i], fontsize=14, fontweight='bold')
        axes[i].axis('off')
        
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "pipeline_steps_visual.png"))
    plt.show()

In [ ]:
def calculate_distance(p1, p2):
    return np.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

def calculate_speed(distance, fps):
    return (distance * fps) * 3.6

def run_pipeline_on_video(input_path, use_pipeline=True):
    import math
    cap = cv2.VideoCapture(input_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    mode_name = "CASCADED" if use_pipeline else "BASELINE"
    vid_name = os.path.basename(input_path)
    
    out_vid = os.path.join(OUTPUT_DIR, f"result_{mode_name}_{vid_name}")
    out_csv = os.path.join(OUTPUT_DIR, f"log_{mode_name}_{vid_name}.csv")
    
    writer = cv2.VideoWriter(out_vid, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    
    csv_file = open(out_csv, 'w', newline='')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(["Frame", "TrackID", "Speed_kmh", "Conf", "Processing_ms"])
    
    tracker = DeepSort(max_age=50)
    model = YOLO("yolov8n.pt") 
    prev_positions = {}
    speed_accumulator = {}
    frame_count = 0
    
    pts = SOURCE_POLYGON.astype(np.int32).reshape((-1, 1, 2))
    polygon_mask = np.zeros((height, width), dtype=np.uint8)
    cv2.fillPoly(polygon_mask, [pts], 255)
    
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        start = time.time()
        
        if use_pipeline:
            processed = apply_cascaded_pipeline(frame)
        else:
            processed = frame # Baseline
            
        results = model(processed, verbose=False)
        proc_ms = (time.time() - start) * 1000
        
        detections = []
        confidences = []
        for pred in results:
            for box in pred.boxes:
                if float(box.conf[0]) > CONF_THRESHOLD:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    # ROI Check
                    cx, cy = (x1+x2)//2, (y1+y2)//2
                    if polygon_mask[cy, cx] == 255:
                        detections.append([[x1, y1, x2-x1, y2-y1], float(box.conf[0]), int(box.cls[0])])
                        confidences.append(float(box.conf[0]))
        
        avg_conf = sum(confidences)/len(confidences) if confidences else 0
        tracks = tracker.update_tracks(detections, frame=processed)
        
        for track in tracks:
            if not track.is_confirmed(): continue
            tid = track.track_id
            
            center = np.array([[(track.to_ltrb()[0]+track.to_ltrb()[2])//2, (track.to_ltrb()[1]+track.to_ltrb()[3])//2]], dtype=np.float32)
            trans = cv2.perspectiveTransform(center[None, :, :], TRANSFORM_MATRIX)
            
            spd = 0
            if tid in prev_positions:
                dist = calculate_distance(prev_positions[tid], trans[0][0])
                spd = calculate_speed(dist, fps)
                if tid not in speed_accumulator: speed_accumulator[tid] = []
                speed_accumulator[tid].append(spd)
                if len(speed_accumulator[tid]) > 5: speed_accumulator[tid].pop(0)
            prev_positions[tid] = trans[0][0]
            
            avg_spd = sum(speed_accumulator[tid])/len(speed_accumulator[tid]) if tid in speed_accumulator else 0
            csv_writer.writerow([frame_count, tid, avg_spd, avg_conf, proc_ms])
            
            # Draw
            x1, y1, x2, y2 = map(int, track.to_ltrb())
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"{avg_spd:.0f}km/h", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            
        writer.write(frame)
        frame_count += 1
        if frame_count % 50 == 0: print(f"  > Frame {frame_count} done.")
        
    cap.release()
    writer.release()
    csv_file.close()
    print("Done.")

run_pipeline_on_video(INPUT_VIDEO_PATH, use_pipeline=True)

In [ ]:
def run_batch_pipeline():
    valid_exts = ('.avi')
    all_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_exts)]
    source_videos = [f for f in all_files if "result_" not in f and "gradient_" not in f]
    
    for vid in source_videos:
        path = os.path.join(INPUT_DIR, vid)
        
        if not os.path.exists(os.path.join(OUTPUT_DIR, f"log_BASELINE_{vid}.csv")):
            run_pipeline_on_video(path, use_pipeline=False)
            
        if not os.path.exists(os.path.join(OUTPUT_DIR, f"log_CASCADED_{vid}.csv")):
            run_pipeline_on_video(path, use_pipeline=True)

run_batch_pipeline()

In [ ]:
def generate_comparison_report():
    logs = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("log_") and f.endswith(".csv")]
    data = []
    for log in logs:
        parts = log.replace(".csv", "").split("_")
        mode = parts[1] 
        vid_name = "_".join(parts[2:])
        
        df = pd.read_csv(os.path.join(OUTPUT_DIR, log))
        if df.empty: continue
            
        # Metrics
        unique_ids = df['TrackID'].nunique()
        avg_dur = df.groupby('TrackID')['Frame'].count().mean()
        avg_conf = df['Conf'].mean()
        
        data.append({
            'Video': vid_name,
            'Method': mode,
            'Stability (IDs)': unique_ids,
            'Avg Duration': avg_dur,
            'Confidence': avg_conf
        })
        
    df_res = pd.DataFrame(data)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    sns.barplot(x='Method', y='Stability (IDs)', data=df_res, ax=axes[0], palette=['gray', 'green'])
    axes[0].set_title("Track Fragmentation (Lower is Better)")
    
    sns.barplot(x='Method', y='Avg Duration', data=df_res, ax=axes[1], palette=['gray', 'green'])
    axes[1].set_title("Track Duration (Higher is Better)")
    
    sns.barplot(x='Method', y='Confidence', data=df_res, ax=axes[2], palette=['gray', 'green'])
    axes[2].set_title("AI Confidence (Higher is Better)")
    
    plt.suptitle("Baseline vs. Cascaded Pipeline Performance", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "final_pipeline_comparison.png"))
    plt.show()
    
    print("Report Generated.")
    print(df_res.groupby('Method').mean())

generate_comparison_report()